In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create Ordinary Least Squares (OLS) linear regression from scratch

What OLS does is fits the following linear equation onto a scatter of points: $$y=\beta_0+\beta_1x+\epsilon$$
Where:
- $\beta_0$ is the y-intercept
- $\beta_1$ is the slope of the line
- $\epsilon$ is the residual error
This residual can be found from: $$\epsilon=y_i-\hat{y}_i=y_i-(\beta_0+\beta_1x)$$
The goal is to find the coefficents that minimize the Sum of Squared Residuals (SSR). The residuals are squared because otherwise the negative and positive residuals would cancel $$S(\beta_0, \beta_1) = \sum^n_{i=1}\epsilon^2_i = (y_i-(\beta_0+\beta_1x))^2$$
To find the minimum of this function, we  take the partial derivatives of $S$ with respect to $\beta_0$ and $\beta_1$, set them equal to zero, and solve the system of equations. But there has to be an easier way to do this on a computer? Thankfully there is a closed form solution we can use in matrix form, where the equation looks like: $$Y=X\beta+\epsilon$$
- $Y$ is a vector of the form: $n\times 1$
- $X$ is a matrix of the form: $n\times k$ with a column of 1s added to handle the intercept
- $\beta$ is a vector of coefficients to solve for of the form: $k\times 1$
The form for the $\beta$ values is: $$\beta=(X^TX)^{-1}X^TY$$

In [5]:
# finding beta vector
def OLS(X, Y):
    return np.linalg.inv(X.T @ X) @ X.T @ Y

# making X and Y suitable for OLS
def construct_XY(x,y):
    X = np.column_stack((np.ones(len(x)), x))
    Y = np.asarray(y)
    return X, Y

# find y_pred
def make_preds(X, beta):
    return X @ beta

But how can we check how well our regressor is doing? We can use bias, variance, and $R^2$ score
- Bias measures how far off a model's average predictions are from the true values. In other words, it measures how much a model tends to oversimplify or underfit.
- Variance measures how much the predictions fluctuate based on different training data. In other words, it measures how sensitive the model is, or overfitting.
- $R^2$ score is a metric that represents the proportion of variance in the target variable that can be explained by the model. If this value is too high on training data it can mean overfitting
- F-statistic: The F-statistic tests the global null hypothesis that all your regression coefficients are equal to zero. Basically, how much better is the fit than a straight line

You'll notice that it will take more than one dataset to compute these, so to implement bias and variance, we can use a Monte Carlo simulation. We need to test multiple datasets to do this, so we can slightly randomize the data (add noise) to the data every simulation to determine these values

In [6]:
def R_2(y_true, y_pred):
    # sum of residuals
    ss_res = np.sum((y_true - y_pred) ** 2)

    # sum of squares
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    return 1 - (ss_res/ss_tot)

def F_stat(y_true, y_pred, p):
    n = len(y_true)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    ss_reg = ss_tot - ss_res
    
    df_reg = p
    df_res = n - p - 1
    
    numerator = ss_reg / df_reg
    denominator = ss_res / df_res
    
    return numerator / denominator
def LL(y_true, y_pred):
    n = len(y_true)
    ss_res = np.sum((y_true - y_pred) ** 2)
    
    sigma_sq_mle = ss_res / n
    
    term1 = - (n / 2) * np.log(2 * np.pi)
    term2 = - (n / 2) * np.log(sigma_sq_mle)
    term3 = - (n / 2)
    
    return term1 + term2 + term3

In [7]:
# Test + Monte Carlo bias + variance calculation

rng = np.random.default_rng(42)

x = np.linspace(0, 30, 20)
true_slope, true_int = 7, 9
y = lambda xi: true_slope*xi + true_int

n_sims = 100
noise_range = 2
all_preds = np.zeros((n_sims, len(x)))

# do fit testing
for i in range(n_sims):
    noise = rng.uniform(low=-noise_range, high=noise_range, size=x.shape)
    noisy_data = y(x) + noise

    X, Y = construct_XY(x, noisy_data)
    beta = OLS(X, Y)

    all_preds[i] = make_preds(X, beta)

y_true = y(x)

mean_pred = all_preds.mean(axis=0)
bias_sq = (mean_pred - y_true) ** 2
variance = all_preds.var(axis=0)
R2 = R_2(y_true, mean_pred)
F_s = F_stat(y_true, mean_pred, 1)
LL_ = LL(y_true, mean_pred)
mse = np.mean((all_preds - y_true) ** 2, axis=0)

print(f"Variance: {variance}")
print(f"Bias (squared): {bias_sq}")
print(f"MSE: {mse}")
print(f"R^2 value: {R2}")
print(f"F statistic: {F_s}")
print(f"Log Likelihood: {LL_}")

Variance: [0.2890216  0.24371736 0.20322219 0.16753611 0.1366591  0.11059118
 0.08933233 0.07288257 0.06124189 0.05441029 0.05238776 0.05517432
 0.06276996 0.07517468 0.09238848 0.11441136 0.14124332 0.17288436
 0.20933448 0.25059368]
Bias (squared): [8.01391093e-04 5.82477223e-04 3.98412696e-04 2.49197512e-04
 1.34831671e-04 5.53151729e-05 1.06480183e-05 8.30206752e-07
 2.58617384e-05 8.57426131e-05 1.80472831e-04 3.10052392e-04
 4.74481296e-04 6.73759544e-04 9.07887134e-04 1.17686407e-03
 1.48069034e-03 1.81936596e-03 2.19289093e-03 2.60126523e-03]
MSE: [0.28982299 0.24429983 0.2036206  0.1677853  0.13679393 0.11064649
 0.08934298 0.0728834  0.06126775 0.05449603 0.05256824 0.05548437
 0.06324444 0.07584844 0.09329637 0.11558822 0.14272401 0.17470373
 0.21152737 0.25319495]
R^2 value: 0.9999998256648424
F statistic: 103249379.54181278
Log Likelihood: 44.1501726033284


In [15]:
!pip install statsmodels

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 34.0 MB/s eta 0:00:0000:01

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [8]:
# Compare with statsmodels
import statsmodels.api as sm

df = pd.DataFrame({'x': x, 'y': y(x)})
X_with_constant = sm.add_constant(df['x'])
model = sm.OLS(df['y'], X_with_constant)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 4.244e+32
Date:                Fri, 12 Jun 2026   Prob (F-statistic):          8.24e-284
Time:                        10:51:17   Log-Likelihood:                 610.91
No. Observations:                  20   AIC:                            -1218.
Df Residuals:                      18   BIC:                            -1216.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          9.0000   5.96e-15   1.51e+15      0.0